# Test the fine-tuned Qwen2.5-7B: generate new MCQs for chosen topics

**Kaggle:** Add Input → Your Work → the fine-tuning notebook (its output has `final_adapter`). The dataset is not needed.
Settings: GPU T4 x2, Internet ON. Then Run All (interactive is fine, about 25 minutes: fine-tuned, then untrained, on the same 18 topics).

**Colab:** put `qwen7b_mcq_adapter.zip` in Google Drive folder `adaptive-quiz`, choose a T4 GPU runtime, Run All.

Each MCQ goes through the checker (4 distinct options, number consistency, step-by-step re-solve). The output shows
`verify: PASS/fail` for each one, and everything is saved to `my_topics_mcqs.csv`.

In [ ]:
!pip install -q unsloth

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
from unsloth import FastLanguageModel
import glob, json, re, random, time, zipfile
import pandas as pd, torch
OUT_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "/content"
if not os.path.exists("/kaggle/input"):        # Colab: the adapter zip lives in Drive folder adaptive-quiz
    from google.colab import drive
    drive.mount("/content/drive")
hits = glob.glob("/kaggle/input/**/final_adapter/adapter_config.json", recursive=True)
if not hits:                                    # only the zip was added: unzip it
    for z in glob.glob("/kaggle/input/**/qwen7b_mcq_adapter.zip", recursive=True) + glob.glob("/content/drive/MyDrive/adaptive-quiz/qwen7b_mcq_adapter.zip"):
        zipfile.ZipFile(z).extractall(f"{OUT_DIR}/final_adapter")
    hits = glob.glob(f"{OUT_DIR}/final_adapter/**/adapter_config.json", recursive=True)
assert hits, "fine-tuned adapter not found: add the fine-tuning notebook's output as input"
ADAPTER_DIR = os.path.dirname(hits[0])
CFG = json.load(open(f"{ADAPTER_DIR}/train_config.json"))
SYSTEM = CFG["system"]
print("adapter:", ADAPTER_DIR)
model, tok = FastLanguageModel.from_pretrained(ADAPTER_DIR, max_seq_length=CFG["max_len"], load_in_4bit=True, dtype=None)
FastLanguageModel.for_inference(model)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
print(type(model).__name__, "| fine-tuned adapter loaded:", hasattr(model, "peft_config"))

def prompt_text(input_text, shots=()):
    m = [{"role": "system", "content": SYSTEM}]
    for s in shots:
        m += [{"role": "user", "content": s["input_text"]}, {"role": "assistant", "content": s["target_text"]}]
    m.append({"role": "user", "content": input_text})
    return tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True)

In [ ]:
import re

LETTERS = "ABCD"


def build_input(branch, subject, topic, difficulty, context):
    return (f"generate mcq | branch: {branch} | subject: {subject} | "
            f"topic: {topic} | difficulty: {difficulty} | context: {context}")


_PATTERN = re.compile(
    r"question:\s*(?P<q>.+?)\s*options:\s*"
    r"\(A\)\s*(?P<A>.+?)\s*\(B\)\s*(?P<B>.+?)\s*\(C\)\s*(?P<C>.+?)\s*\(D\)\s*(?P<D>.+?)\s*"
    r"answer:\s*(?P<ans>[ABCD])\b\s*(?:explanation:\s*(?P<exp>.*))?$",
    re.S)


def parse_mcq(text):
    """Parse model output into a dict, or return None if the format is broken."""
    m = _PATTERN.search(text.strip())
    if not m:
        return None
    options = [m.group(l).strip() for l in LETTERS]
    norm = [re.sub(r"\s+", " ", o.lower()) for o in options]
    if any(not o for o in options) or len(set(norm)) < 4:
        return None
    return {"question": m.group("q").strip(), "options": options,
            "answer": m.group("ans"), "explanation": (m.group("exp") or "").strip()}

In [ ]:
NUM = r"-?\d+(?:\.\d+)?"
def numeric_conflict(p):
    # explanation's final number equals a different option than the key -> wrong key
    nums = re.findall(NUM, p["explanation"].replace(",", ""))
    if not nums: return False
    last = float(nums[-1])
    hits = [i for i, o in enumerate(p["options"])
            if re.fullmatch(r"\s*" + NUM + r"\s*\S{0,8}\s*", o.replace(",", "")) and
               abs(float(re.findall(NUM, o.replace(",", ""))[0]) - last) < 1e-6]
    return bool(hits) and "ABCD"[hits[0]] != p["answer"]

def verify_prompt(p):
    q = p["question"] + "\n" + "\n".join(f"({l}) {o}" for l, o in zip("ABCD", p["options"]))
    m = [{"role": "user", "content": "Solve this multiple-choice question. Think step by step and check any calculation. "
                                     "If no option or more than one option is correct, say so. "
                                     "End with a last line exactly like: Final answer: <letter or NONE>\n\n" + q}]
    return tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def lm(prompts, n=1, sample=True, max_new=320):
    tok.padding_side = "left"
    enc = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
    kw = dict(do_sample=True, temperature=0.7, top_p=0.9) if sample else dict(do_sample=False)
    ids = model.generate(**enc, max_new_tokens=max_new, num_return_sequences=n, pad_token_id=tok.pad_token_id, **kw)
    return tok.batch_decode(ids[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)

def generate_checked(rows, shots=(), n_cand=CFG["n_cand"], bs=2):
    out = []
    for i in range(0, len(rows), bs):
        chunk = rows[i:i+bs]
        texts = lm([prompt_text(r["input_text"], shots) for r in chunk], n=n_cand)
        for j, r in enumerate(chunk):
            cands = [p for p in map(parse_mcq, texts[j*n_cand:(j+1)*n_cand]) if p]
            res = {"valid": bool(cands), "passed_numeric": False, "passed_verify": False, "mcq": None, "verify_note": ""}
            cands_num = [p for p in cands if not numeric_conflict(p)]
            res["passed_numeric"] = bool(cands_num)
            if cands_num:
                ver = lm([verify_prompt(p) for p in cands_num], sample=False, max_new=400)
                for p, v in zip(cands_num, ver):
                    m = re.search(r"Final answer:\s*\(?([A-D]|NONE)", v)
                    if m and m.group(1) == p["answer"]:
                        res.update(passed_verify=True, mcq=p, verify_note=v[-300:]); break
                if not res["mcq"]:
                    res.update(mcq=cands_num[0], verify_note=ver[0][-300:])
            elif cands:
                res["mcq"] = cands[0]
            out.append(res)
    return out

def to_rows(rows, gens, label):
    recs = []
    for r, g in zip(rows, gens):
        p = g["mcq"]
        recs.append({"model": label, "branch": r["branch"], "subject": r["subject"], "topic": r["topic"],
                     "difficulty": r["difficulty"], "valid": g["valid"], "passed_numeric": g["passed_numeric"],
                     "passed_verify": g["passed_verify"],
                     "mcq": None if not p else f"Q: {p['question']}\n" + "\n".join(f"  ({l}) {o}" for l, o in zip("ABCD", p["options"]))
                            + f"\n  answer: {p['answer']}  |  {p['explanation']}",
                     "verify_note": g["verify_note"], "is_correct (you fill Y/N)": ""})
    return recs

## Stronger number check (replaces the one above): the explanation's final number must match the marked option

In [ ]:
import re
NUM = r"-?\d+(?:\.\d+)?(?:e-?\d+)?"
def nums(s): return [float(x) for x in re.findall(NUM, s.replace(",", ""))]
def option_value(o):
    o = o.replace(",", "").strip()
    m = re.fullmatch(r"(?:about |approximately |~)?(" + NUM + r")\s*(?:[a-zA-Z%° ]{0,12})", o, re.I)
    return float(m.group(1)) if m else None
def numeric_conflict(p):
    vals = [option_value(o) for o in p["options"]]
    key = vals["ABCD".index(p["answer"])]
    if key is None or sum(v is not None for v in vals) < 3: return False     # not a numeric question
    found = nums(p["explanation"])
    if not found: return False
    last = found[-1]
    close = lambda a, b: abs(a - b) <= 0.02 * max(abs(a), abs(b), 1e-9)
    pct = any(t in p["options"]["ABCD".index(p["answer"])].lower() for t in ("%", "percent"))
    ok = close(last, key) or (pct and close(last * 100, key))
    return not ok                        # explanation's final result must equal the keyed option

## Topics (from the dataset). Edit or add your own in the same format.

In [ ]:
TOPICS = [
 {
  "branch": "DSAI",
  "subject": "Machine Learning",
  "topic": "Evaluation Metrics",
  "context": "Classification models are evaluated with metrics derived from the confusion matrix, which counts true positives (TP), false positives (FP), true negatives (TN) and false negatives (FN). Accuracy = (TP + TN) / total is misleading on imbalanced data. Precision = TP / (TP + FP) is the fraction of predicted positives that are correct, important when false alarms are costly. Recall, or sensitivity, = TP / (TP + FN) is the fraction of actual positives found, important when missing positives is costly, such as in disease screening. The F1 score is the harmonic mean, F1 = 2 * P * R / (P + R). Specificity = TN / (TN + FP). The ROC curve plots the true positive rate against the false positive rate across thresholds, and the area under it (AUC) summarises ranking quality, with 0.5 for random guessing. Regression is evaluated with MAE, MSE, RMSE and R^2."
 },
 {
  "branch": "DSAI",
  "subject": "Multivariate Analysis",
  "topic": "Quadratic Discriminant Analysis",
  "context": "Quadratic discriminant analysis (QDA) assumes that each class k is multivariate normal with its own mean mu_k and its own covariance matrix Sigma_k. The discriminant score is delta_k(x) = -0.5 * log det(Sigma_k) - 0.5 * (x - mu_k)^T Sigma_k^-1 (x - mu_k) + log(pi_k), and x is assigned to the class with the highest score. Because the quadratic terms differ between classes, the decision boundaries are quadratic surfaces. QDA must estimate K separate covariance matrices, each with p(p+1)/2 parameters, whereas LDA estimates only one. For example, with p = 10 and K = 3, QDA needs 3 * 55 = 165 covariance parameters while LDA needs 55. QDA is therefore more flexible but has higher variance, so it works best when class covariances truly differ and training data are plentiful; with small samples LDA often generalizes better. If all Sigma_k are equal, QDA reduces to LDA."
 },
 {
  "branch": "CSE",
  "subject": "Theory of Computation",
  "topic": "Reducibility",
  "context": "A reduction converts one problem into another so that a solution to the second yields a solution to the first. Language A is mapping reducible to B, written A <=m B, if there is a computable function f such that w is in A if and only if f(w) is in B. If A <=m B and B is decidable, then A is decidable; equivalently, if A is undecidable and A <=m B, then B is undecidable. Likewise, if A <=m B and B is Turing-recognizable, then A is recognizable. A standard example reduces A_TM to HALT_TM: given <M, w>, build M' that runs M on w and loops forever if M rejects, so M' halts on w exactly when M accepts w. Similar reductions show that E_TM and EQ_TM are undecidable. Also, A <=m B if and only if complement(A) <=m complement(B)."
 },
 {
  "branch": "CSE",
  "subject": "Computer Graphics",
  "topic": "Cohen-Sutherland Clipping",
  "context": "The Cohen-Sutherland algorithm clips lines against a rectangular window by giving each endpoint a 4-bit region code, or outcode. Using the bit order top, bottom, right, left, a bit is 1 if the point is above ymax, below ymin, right of xmax or left of xmin, respectively, so a point inside the window has code 0000 and the plane is divided into nine regions. A line is trivially accepted if both codes are 0000, that is, if code1 OR code2 is 0000. It is trivially rejected if code1 AND code2 is not 0000, because both endpoints then lie outside the same window edge. Otherwise the algorithm picks an outside endpoint, computes the line's intersection with a window edge for one of its set bits, replaces the endpoint with the intersection, recomputes its code and repeats. It is efficient when most lines are fully inside or far outside."
 },
 {
  "branch": "ECE",
  "subject": "Wireless Communication",
  "topic": "Frequency Reuse",
  "context": "Frequency reuse is the use of the same set of radio channels in cells separated by a sufficient distance so that co-channel interference stays acceptable. For hexagonal geometry, valid cluster sizes satisfy N = i^2 + i*j + j^2 where i and j are non-negative integers, giving N = 1, 3, 4, 7, 9, 12, 13 and so on. To find the nearest co-channel cell, move i cells along a chain of hexagons, turn 60 degrees, and move j cells. The co-channel reuse ratio is Q = D/R = sqrt(3*N), where D is the distance between co-channel cell centres and R is cell radius. A larger N increases D/R and reduces interference but lowers capacity, since fewer channels are available per cell; a smaller N increases capacity but raises interference. The frequency reuse factor is 1/N."
 },
 {
  "branch": "ECE",
  "subject": "Radar Systems",
  "topic": "Maximum Unambiguous Range",
  "context": "The maximum unambiguous range of a pulse radar is the greatest range from which an echo returns before the next pulse is transmitted: Run = c / (2 * PRF) = c * T / 2. An echo from a target beyond Run arrives after the next pulse, and the radar interprets it as a short-range target; such echoes are called second-time-around or multiple-time-around echoes. The apparent range is the true range minus integer multiples of Run, Rapparent = Rtrue - n * Run. To avoid ambiguity, the PRF is chosen so that Run exceeds the radar's maximum detection range. Ambiguities can be resolved by using several PRFs: the true range is the value consistent with all PRFs, often found using the Chinese remainder theorem. Pulse-to-pulse coding or PRF jitter also exposes ambiguous echoes, because their apparent range changes from pulse to pulse while true echoes stay fixed."
 }
]

In [ ]:
DIFFICULTIES = ["easy", "medium", "hard"]
rows = [{"input_text": build_input(t["branch"], t["subject"], t["topic"], d, " ".join(t["context"].split())),
         "branch": t["branch"], "subject": t["subject"], "topic": t["topic"], "difficulty": d}
        for t in TOPICS for d in DIFFICULTIES]
t0 = time.time()
gens = generate_checked(rows)
print(f"{len(rows)} MCQs in {(time.time()-t0)/60:.1f} min\n")
for r, g in zip(rows, gens):
    p = g["mcq"]
    print("=" * 90, f"\n[{r['difficulty']}] {r['branch']} | {r['subject']} | {r['topic']}   verify: {'PASS' if g['passed_verify'] else 'fail'}")
    if not p:
        print("NO VALID MCQ"); continue
    print(p["question"])
    for l, o in zip("ABCD", p["options"]): print(f"   ({l}) {o}")
    print("   correct:", p["answer"], "->", p["options"]["ABCD".index(p["answer"])])
    print("   explanation:", p["explanation"])
pd.DataFrame(to_rows(rows, gens, "Qwen2.5-7B fine-tuned")).to_csv(f"{OUT_DIR}/my_topics_mcqs.csv", index=False)
print("\nsaved", f"{OUT_DIR}/my_topics_mcqs.csv")

## Compare: the untrained 7B on the same 18 topics
Same base model with the fine-tuning switched off, plus 2 dataset examples in the prompt (few-shot), the same as in the pilot. Same checks.

In [ ]:
SHOTS = [
 {
  "input_text": "generate mcq | branch: CSE | subject: Blockchain Technology | topic: Hyperledger Fabric | difficulty: easy | context: Hyperledger Fabric is an open-source, permissioned blockchain framework hosted by the Linux Foundation and designed for enterprises. Participants have known identities issued by a Membership Service Provider (MSP) using X.509 certificates. Smart contracts are called chaincode and can be written in Go, Java or JavaScript. Fabric uses an execute-order-validate architecture: endorsing peers first simulate a transaction and sign the read-write set, the ordering service (typically using Raft) orders transactions into blocks, and committing peers validate endorsements against the endorsement policy and check for conflicts before updating the ledger. Channels create separate ledgers for subsets of members, and private data collections keep sensitive data among authorised organizations. Fabric has no native cryptocurrency and its consensus is modular. The ledger consists of the blockchain plus a world state database such as LevelDB or CouchDB.",
  "target_text": "question: Which component issues identities to participants in Fabric? options: (A) Membership Service Provider (B) Ordering service (C) World state database (CouchDB) (D) Endorsing peer answer: A explanation: The MSP issues and manages X.509 identities."
 },
 {
  "input_text": "generate mcq | branch: DSAI | subject: Deep Learning | topic: LSTM | difficulty: hard | context: Long short-term memory (LSTM) networks are RNNs designed to learn long-range dependencies. Besides the hidden state h_t, an LSTM keeps a cell state c_t. Three sigmoid gates control information flow: the forget gate f_t decides what to erase from the cell state, the input gate i_t decides what new information to write, and the output gate o_t decides what part of the cell state to expose. cell is updated additively: c_t = f_t * c_(t-1) + i_t * g_t, with h_t = o_t * tanh(c_t). Because the cell update is additive and gated, gradients can flow across many time steps when the forget gate stays near 1, which mitigates vanishing gradients. An LSTM layer has four sets of weights, one per gate and one for the candidate, so it has about four times the parameters of a vanilla RNN of the same size.",
  "target_text": "question: If c_t = 0 and o_t = 0.9, what is the hidden state h_t? options: (A) 0 (B) 0.45 (C) 0.9 (D) 1 answer: A explanation: h_t = o_t * tanh(c_t) = 0.9 * tanh(0) = 0."
 }
]

In [ ]:
t0 = time.time()
with model.disable_adapter():
    base_gens = generate_checked(rows, shots=SHOTS)
print(f"untrained: {len(rows)} MCQs in {(time.time()-t0)/60:.1f} min\n")
def show(p):
    if not p: return "NO VALID MCQ"
    return (p["question"] + "\n" + "\n".join(f"   ({l}) {o}" for l, o in zip("ABCD", p["options"]))
            + f"\n   correct: {p['answer']} -> {p['options']['ABCD'.index(p['answer'])]}\n   explanation: {p['explanation']}")
for r, g_ft, g_b in zip(rows, gens, base_gens):
    print("#" * 90, f"\n[{r['difficulty']}] {r['branch']} | {r['subject']} | {r['topic']}")
    print(f"--- FINE-TUNED  verify: {'PASS' if g_ft['passed_verify'] else 'fail'}  numbers: {'ok' if g_ft['passed_numeric'] else 'MISMATCH'}")
    print(show(g_ft["mcq"]))
    print(f"--- UNTRAINED   verify: {'PASS' if g_b['passed_verify'] else 'fail'}  numbers: {'ok' if g_b['passed_numeric'] else 'MISMATCH'}")
    print(show(g_b["mcq"]))
both = pd.DataFrame(to_rows(rows, gens, "7B fine-tuned") + to_rows(rows, base_gens, "7B untrained (few-shot)"))
both.to_csv(f"{OUT_DIR}/compare_18_topics.csv", index=False)
print("\nsaved", f"{OUT_DIR}/compare_18_topics.csv")
print(both.groupby("model")[["valid", "passed_numeric", "passed_verify"]].mean().round(2))

## Your own input

In [ ]:
BRANCH, SUBJECT, TOPIC = "CSE", "Operating Systems", "Deadlock"
CONTEXT = """A deadlock is a situation where a set of processes are blocked because each process is holding a resource
and waiting for another resource held by some other process. Four conditions must hold at the same time for a deadlock:
mutual exclusion, hold and wait, no preemption and circular wait. Deadlocks can be handled by prevention, which breaks one
of the four conditions, by avoidance, such as the Banker's algorithm, which grants a request only if the system stays in a
safe state, or by detection and recovery, where the system finds cycles in the wait-for graph and then aborts a process or
preempts resources."""
mine = [{"input_text": build_input(BRANCH, SUBJECT, TOPIC, d, " ".join(CONTEXT.split())), "difficulty": d} for d in DIFFICULTIES]
for r, g in zip(mine, generate_checked(mine)):
    p = g["mcq"]; print(f"\n[{r['difficulty']}] verify: {'PASS' if g['passed_verify'] else 'fail'}")
    if p:
        print(p["question"]); [print(f"   ({l}) {o}") for l, o in zip("ABCD", p["options"])]
        print("   correct:", p["answer"], "->", p["options"]["ABCD".index(p["answer"])], "|", p["explanation"])